# P10.6-AI - RSNA dataset preflight

Preflight train-only para RSNA LumbarDISC. Este notebook prepara inventario, distribuciones y reportes sanitizados para clasificacion asistida de hallazgo candidato. No entrena, no accede al test oficial y no genera diagnostico clinico.

## Guardias operativas

- `humanReviewRequired=true` y `notClinicalDiagnosis=true`.
- El conjunto oficial de test se ignora completamente si existe.
- Protrusion y extrusion no se entrenan con RSNA.
- Los reportes se escriben fuera de Git bajo `PFI_P10_6_OUTPUT_ROOT`.

In [ ]:
# 1) Dependencias
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_MODULES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pydicom": "pydicom",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "monai": "monai",
    "yaml": "pyyaml",
}

missing = [package for module, package in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception as exc:
        raise RuntimeError(f"Dependencias faltantes fuera de Colab: {missing}") from exc
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [ ]:
# 2) Imports, versiones y semillas
import json
import os
from pathlib import Path

import pandas as pd
import pydicom
import SimpleITK as sitk
import torch
import yaml

from sklearn.model_selection import StratifiedGroupKFold  # noqa: F401

print({
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cudaAvailable": torch.cuda.is_available(),
    "cudaVersion": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "pydicom": pydicom.__version__,
    "SimpleITK": sitk.Version_VersionString(),
})


In [ ]:
# 3) Montaje opcional de Google Drive
USE_GOOGLE_DRIVE = os.getenv("PFI_USE_GOOGLE_DRIVE", "1").strip() == "1"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception:
        print("Google Drive no disponible; continuo con rutas locales/configuradas.")


In [ ]:
# 4) Repositorio y modulo AI
PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
PFI_REPO_REF = os.getenv("PFI_REPO_REF", "enzo/p10-6-ai-rsna-findings")

if not (PFI_REPO_ROOT / "ai_service" / "pfi_ai_service").exists():
    if os.getenv("PFI_ALLOW_REPO_CLONE", "0") != "1":
        raise RuntimeError("PFI_REPO_ROOT no contiene AI Module; definir PFI_ALLOW_REPO_CLONE=1 para clonar.")
    subprocess.check_call(["git", "clone", PFI_REPO_URL, str(PFI_REPO_ROOT)])

subprocess.check_call(["git", "fetch", "origin"], cwd=PFI_REPO_ROOT)
subprocess.check_call(["git", "checkout", PFI_REPO_REF], cwd=PFI_REPO_ROOT)

sys.path.insert(0, str(PFI_REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_preflight import (
    build_config,
    run_preflight,
    runtime_versions,
    set_reproducible_seed,
    validate_dataset_structure,
)


In [ ]:
# 5) Configuracion reproducible
CFG = build_config()
set_reproducible_seed(CFG.seed)
print(json.dumps({
    "seed": CFG.seed,
    "syntheticMode": CFG.synthetic,
    "rsnaRootConfigured": str(CFG.rsna_root),
    "outputRootConfigured": str(CFG.output_root),
    "modelRootConfigured": str(CFG.model_root),
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}, indent=2))


In [ ]:
# 6) Preflight de estructura train-only
if not CFG.synthetic:
    structure = validate_dataset_structure(CFG)
else:
    structure = {"syntheticMode": True, "officialTestAccessed": False}

if structure.get("officialTestAccessed") is not False:
    raise RuntimeError("El preflight no puede acceder al test oficial.")

print(json.dumps(structure, indent=2))


In [ ]:
# 7) Ejecutar inventario y reportes externos
summary = run_preflight(CFG)
if summary["officialTestAccessed"] is not False:
    raise RuntimeError("officialTestAccessed debe permanecer false.")
if not summary["humanReviewRequired"] or not summary["notClinicalDiagnosis"]:
    raise RuntimeError("Gobernanza invalida para P10.6-AI.")
print(json.dumps({
    "nStudies": summary["nStudies"],
    "nSeries": summary["nSeries"],
    "nDicom": summary["nDicom"],
    "officialTestPresent": summary["officialTestPresent"],
    "officialTestAccessed": summary["officialTestAccessed"],
    "outputs": summary["outputs"],
}, indent=2))


In [ ]:
# 8) Evidencia sanitizada para Notebook 54
print(json.dumps({
    "dataset": summary["dataset"],
    "csvSha256": summary["csvSha256"],
    "sequenceAvailability": summary["sequenceAvailability"],
    "coordinateIssues": summary["coordinateIssues"],
    "futureMetrics": summary["futureMetrics"],
    "limitations": summary["limitations"],
    "nextNotebook": "54_internal_split_and_model_plan",
}, indent=2))
